# Brain Tumor Segmentation — MedSAM ViT-B + Transformer U-Net Decoder (PyTorch)

This experiment changes the architecture substantially while keeping the **positive-only slice protocol** used by the stronger DenseNet baseline, so the comparison is meaningful.

### Main idea
- Use **MedSAM ViT-B** pretrained on medical image segmentation as the transformer encoder.
- Do **transfer learning** rather than training a transformer from scratch.
- Freeze the MedSAM encoder first and train a lightweight decoder.
- Then fine-tune only the last transformer blocks + decoder.
- Use a **pyramid Transformer-U-Net decoder** with attention gates.
- Keep Focal Tversky (`alpha=0.7`, `gamma=0.75`) initially.

MedSAM itself is a promptable medical segmentation foundation model. Its official implementation uses a ViT-B image encoder and a prompt/mask decoder; its standard inference expects a bounding-box prompt. Here we deliberately use the **pretrained MedSAM image encoder as the transfer-learning backbone** and attach an automatic segmentation decoder, so the final model does not need the ground-truth box at inference. This avoids using the test mask as a prompt. The official MedSAM repository documents the ViT-B checkpoint and its box-prompted inference/training setup. citeturn0search0turn1search5

## Why this experiment

The previous Swin-Tiny + U-Net model reached about Dice **0.821** on the positive-only protocol, below the DenseNet121 result (~0.858). Rather than simply increasing the Swin size, this experiment tests a different pretrained representation that was explicitly trained on medical images.

A published brain-tumor study also found that adapting SAM's mask decoder with transfer learning improved over vanilla pretrained SAM, although nnU-Net remained stronger in that particular experiment. That makes **transfer learning + controlled fine-tuning** a reasonable experiment, not a guaranteed improvement. citeturn0academia12

In [1]:
# Kaggle setup
import os, sys, glob, random, math, subprocess, importlib.util
from pathlib import Path

# Install the official SAM implementation if it is not already available.
if importlib.util.find_spec("segment_anything") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/facebookresearch/segment-anything.git"
    ])

if importlib.util.find_spec("huggingface_hub") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from skimage import io
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from segment_anything import sam_model_registry
from huggingface_hub import hf_hub_download

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


PyTorch: 2.10.0+cu128
Device: cuda
GPU: Tesla T4


## 1. Load the LGG MRI dataset

In [2]:
CANDIDATE_ROOTS = [
    '/kaggle/input/lgg-mri-segmentation/kaggle_3m',
    '../input/lgg-mri-segmentation/kaggle_3m',
]
DATA_ROOT = next((p for p in CANDIDATE_ROOTS if os.path.exists(p)), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Could not find /kaggle/input/lgg-mri-segmentation/kaggle_3m')

rows = []
for patient_dir in sorted(glob.glob(os.path.join(DATA_ROOT, '*'))):
    if not os.path.isdir(patient_dir):
        continue
    patient_id = os.path.basename(patient_dir)
    for path in glob.glob(os.path.join(patient_dir, '*.tif')):
        rows.append((patient_id, path))

df = pd.DataFrame(rows, columns=['patient_id', 'path'])
df_imgs = df[~df['path'].str.contains('mask', case=False, regex=False)].copy()
df_masks = df[df['path'].str.contains('mask', case=False, regex=False)].copy()
mask_map = {Path(p).stem.replace('_mask', ''): p for p in df_masks['path']}

records = []
for image_path in df_imgs['path']:
    stem = Path(image_path).stem
    if stem in mask_map:
        records.append((os.path.basename(os.path.dirname(image_path)), image_path, mask_map[stem]))

brain_df = pd.DataFrame(records, columns=['patient_id', 'image_path', 'mask_path'])

def has_tumor(mask_path):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    return int(mask is not None and mask.max() > 0)

brain_df['mask'] = brain_df['mask_path'].apply(has_tumor)
brain_df = brain_df.reset_index(drop=True)

print('Total paired slices:', len(brain_df))
print(brain_df['mask'].value_counts().rename({0:'tumor-free', 1:'tumor-positive'}))


Total paired slices: 3929
mask
tumor-free        2556
tumor-positive    1373
Name: count, dtype: int64


## 2. Use the same positive-only protocol first

We intentionally keep tumor-positive slices only for this experiment. The previous full-data experiment changed the training distribution substantially, so it is not a clean architecture comparison. We can test tumor-free slices later as a separate ablation.

In [3]:
USE_POSITIVE_ONLY = True
working_df = brain_df[brain_df['mask'] == 1].copy() if USE_POSITIVE_ONLY else brain_df.copy()

X_train, X_val = train_test_split(working_df, test_size=0.15, random_state=SEED, shuffle=True)
X_test, X_val = train_test_split(X_val, test_size=0.5, random_state=SEED, shuffle=True)

X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

print('Train:', len(X_train), 'Val:', len(X_val), 'Test:', len(X_test))
print('Protocol: positive-only slice-level split')


Train: 1167 Val: 103 Test: 103
Protocol: positive-only slice-level split


## 3. Dataset and preprocessing

MedSAM's official image encoder uses a 1024×1024 image representation. We therefore resize each 256×256 MRI slice to 1024×1024 for the encoder, while the decoder produces a final 256×256 segmentation mask. The MRI is normalized per image; masks use nearest-neighbor interpolation.

In [4]:
ENCODER_SIZE = 1024
OUTPUT_SIZE = 256
BATCH_SIZE = 1       # MedSAM ViT-B at 1024 is memory-heavy
NUM_WORKERS = 2

class BrainTumorDataset(Dataset):
    def __init__(self, dataframe, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def _augment(self, img, mask):
        if np.random.rand() < 0.5:
            img = np.fliplr(img).copy(); mask = np.fliplr(mask).copy()
        if np.random.rand() < 0.5:
            angle = np.random.uniform(-12, 12)
            h, w = img.shape[:2]
            M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
            img = cv2.warpAffine(img, M, (w,h), borderMode=cv2.BORDER_REFLECT)
            mask = cv2.warpAffine(mask, M, (w,h), borderMode=cv2.BORDER_REFLECT, flags=cv2.INTER_NEAREST)
        if np.random.rand() < 0.35:
            scale = np.random.uniform(0.95, 1.05)
            h, w = img.shape[:2]
            M = cv2.getRotationMatrix2D((w/2, h/2), 0, scale)
            img = cv2.warpAffine(img, M, (w,h), borderMode=cv2.BORDER_REFLECT)
            mask = cv2.warpAffine(mask, M, (w,h), borderMode=cv2.BORDER_REFLECT, flags=cv2.INTER_NEAREST)
        if np.random.rand() < 0.5:
            img = img * np.random.uniform(0.9, 1.1) + np.random.uniform(-8, 8)
        return img, mask

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = io.imread(row.image_path)
        mask = io.imread(row.mask_path)

        # Ensure mask is grayscale (2D) in case it was saved as RGB
        if mask.ndim == 3:
            mask = mask[:, :, 0]

        # Resize image to ENCODER_SIZE (1024x1024)
        img = cv2.resize(img, (ENCODER_SIZE, ENCODER_SIZE), interpolation=cv2.INTER_LINEAR).astype(np.float32)
        
        # BUG FIX: Resize mask to OUTPUT_SIZE (256x256), NOT ENCODER_SIZE
        mask = cv2.resize(mask, (OUTPUT_SIZE, OUTPUT_SIZE), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 0).astype(np.float32)

        if self.augment:
            img, mask = self._augment(img, mask)

        # Match MedSAM's published inference preprocessing: scale each image to [0, 1].
        img_min, img_max = float(img.min()), float(img.max())
        img = (img - img_min) / max(img_max - img_min, 1e-6)
        if img.ndim == 2:
            img = np.repeat(img[..., None], 3, axis=-1)
        img = np.clip(img, 0.0, 1.0)

        img = torch.from_numpy(img.transpose(2,0,1)).float()
        mask = torch.from_numpy(mask[None,...]).float()
        return img, mask
        
train_loader = DataLoader(BrainTumorDataset(X_train, True), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(BrainTumorDataset(X_val, False), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(BrainTumorDataset(X_test, False), batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

x0, y0 = next(iter(train_loader))
print('Image:', x0.shape, 'Mask:', y0.shape, 'Mask values:', torch.unique(y0))


Image: torch.Size([1, 3, 1024, 1024]) Mask: torch.Size([1, 1, 1024, 1024]) Mask values: tensor([0., 1.])


## 4. MedSAM checkpoint

The official MedSAM ViT-B checkpoint is about 375 MB. A public Hugging Face copy is available; the official project also documents the `medsam_vit_b.pth` checkpoint. If Kaggle cannot download it, add the checkpoint as a Kaggle Dataset and set `LOCAL_MED_SAM_CHECKPOINT` accordingly. citeturn1search0turn0search0

In [5]:
LOCAL_MED_SAM_CHECKPOINT = '/kaggle/input/medsam-checkpoint/medsam_vit_b.pth'
CHECKPOINT_PATH = LOCAL_MED_SAM_CHECKPOINT

if not os.path.exists(CHECKPOINT_PATH):
    print('Local checkpoint not found. Attempting Hugging Face download (~375 MB)...')
    try:
        CHECKPOINT_PATH = hf_hub_download(
            repo_id='GleghornLab/medsam-vit-b',
            filename='medsam_vit_b.pth'
        )
    except Exception as e:
        raise RuntimeError(
            'MedSAM checkpoint is unavailable. Download medsam_vit_b.pth and add it as a Kaggle Dataset, '
            'then update LOCAL_MED_SAM_CHECKPOINT. Original error: ' + str(e)
        )

print('Checkpoint:', CHECKPOINT_PATH)


Local checkpoint not found. Attempting Hugging Face download (~375 MB)...


medsam_vit_b.pth:   0%|          | 0.00/375M [00:00<?, ?B/s]

Checkpoint: /root/.cache/huggingface/hub/models--GleghornLab--medsam-vit-b/snapshots/ff03cc053ebc863668cde3dd9a48497390b9504f/medsam_vit_b.pth


## 5. MedSAM ViT-B encoder + Transformer-U-Net decoder

Instead of using MedSAM's original prompt decoder, this experiment uses its **medical-pretrained ViT-B image encoder** as the backbone and learns an automatic mask decoder.

The decoder receives intermediate ViT features and builds a pyramid by pooling them to different resolutions. Attention gates are used on the skip connections so the decoder can suppress irrelevant regions.

In [6]:
# Load MedSAM / SAM ViT-B architecture with medical-pretrained weights.
medsam = sam_model_registry['vit_b'](checkpoint=CHECKPOINT_PATH).to(DEVICE)

# We only need the image encoder for this experiment.
encoder = medsam.image_encoder
print('Loaded MedSAM ViT-B image encoder.')


Loaded MedSAM ViT-B image encoder.


In [7]:
class AttentionGate(nn.Module):
    def __init__(self, gate_channels, skip_channels, inter_channels):
        super().__init__()
        self.Wg = nn.Sequential(
            nn.Conv2d(gate_channels, inter_channels, 1, bias=False),
            nn.BatchNorm2d(inter_channels)
        )
        self.Wx = nn.Sequential(
            nn.Conv2d(skip_channels, inter_channels, 1, bias=False),
            nn.BatchNorm2d(inter_channels)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(inter_channels, 1, 1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        a = self.relu(self.Wg(g) + self.Wx(x))
        a = self.psi(a)
        return x * a


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU()
        )
    def forward(self, x): return self.block(x)


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, 2)
        self.att = AttentionGate(out_ch, skip_ch, max(out_ch//2, 16))
        self.conv = ConvBlock(out_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        skip = self.att(x, skip)
        return self.conv(torch.cat([x, skip], dim=1))


class MedSAMTransformerUNet(nn.Module):
    def __init__(self, medsam_encoder):
        super().__init__()
        self.encoder = medsam_encoder

        # MedSAM ViT-B transformer block outputs have 768 channels at 64x64.
        # We build a feature pyramid from intermediate transformer representations.
        self.proj4 = nn.Conv2d(768, 256, 1)
        self.proj3 = nn.Conv2d(768, 192, 1)
        self.proj2 = nn.Conv2d(768, 128, 1)
        self.proj1 = nn.Conv2d(768, 96, 1)

        self.bottleneck = ConvBlock(256, 256)
        self.up3 = UpBlock(256, 96, 192)    # 8 -> 16
        self.up2 = UpBlock(192, 128, 128)   # 16 -> 32
        self.up1 = UpBlock(128, 192, 96)  
        self.up0 = nn.Sequential(
            nn.ConvTranspose2d(96, 64, 2, 2),
            ConvBlock(64, 64),              # 64 -> 128
            nn.ConvTranspose2d(64, 32, 2, 2),
            ConvBlock(32, 32),              # 128 -> 256
        )
        self.head = nn.Conv2d(32, 1, 1)

        self._features = {}
        self._hooks = []
        # Four transformer depths give us feature snapshots.
        for idx, block in enumerate(self.encoder.blocks):
            if idx in [2, 5, 8, 11]:
                self._hooks.append(block.register_forward_hook(self._make_hook(idx)))

    def _make_hook(self, idx):
        def hook(module, inputs, output):
            self._features[idx] = output
        return hook

    @staticmethod
    def _nchw(x):
        if x.ndim != 4:
            raise RuntimeError(f'Unexpected ViT feature shape: {tuple(x.shape)}')
        # SAM ViT blocks output [B,H,W,C].
        return x.permute(0,3,1,2).contiguous()

    def forward(self, x):
        self._features = {}
        _ = self.encoder(x)

        f1 = self._nchw(self._features[2])
        f2 = self._nchw(self._features[5])
        f3 = self._nchw(self._features[8])
        f4 = self._nchw(self._features[11])

        # Build a coarse-to-fine pyramid from the transformer representations.
        s1 = self.proj1(F.avg_pool2d(f1, 4))   # 16x16
        s2 = self.proj2(F.avg_pool2d(f2, 2))   # 32x32
        s3 = self.proj3(f3)                    # 64x64
        s4 = self.proj4(f4)                    # 64x64 (deepest feature)

        x = self.bottleneck(F.avg_pool2d(s4, 8))  # 8x8
        x = self.up3(x, s1)                       # 16x16
        x = self.up2(x, s2)                       # 32x32
        x = self.up1(x, s3)                       # 64x64
        x = F.interpolate(
                x,
                size=(128, 128),
                mode='bilinear',
                align_corners=False
            )

        x = self.up0(x)

# Guarantee that prediction and ground-truth mask have identical size.
        x = F.interpolate(
                x,
                size=(OUTPUT_SIZE, OUTPUT_SIZE),
                mode='bilinear',
                align_corners=False
            )

        return self.head(x)


model = MedSAMTransformerUNet(encoder).to(DEVICE)
print('Trainable parameters initially:', sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6, 'M')

# Sanity-check one forward pass.
with torch.no_grad():
    test_logits = model(x0[:1].to(DEVICE))
print('Output:', tuple(test_logits.shape))


Trainable parameters initially: 93.50242 M
Output: (1, 1, 256, 256)


## 6. Loss and training utilities

We keep the same Focal Tversky loss first. This isolates the architecture/pretraining change. Loss tuning comes later only if this model is promising.

In [8]:
def focal_tversky_loss(logits, targets, alpha=0.7, beta=0.3, gamma=0.75, smooth=1.0):
    probs = torch.sigmoid(logits)
    if probs.shape[-2:] != targets.shape[-2:]:
        targets = F.interpolate(targets, size=probs.shape[-2:], mode='nearest')
    targets = targets.float()
    dims = (1,2,3)
    tp = (probs * targets).sum(dims)
    fp = (probs * (1-targets)).sum(dims)
    fn = ((1-probs) * targets).sum(dims)
    tversky = (tp + smooth) / (tp + alpha*fn + beta*fp + smooth)
    return ((1-tversky).clamp_min(0)).pow(gamma).mean()


def dice_from_logits(logits, targets, threshold=0.5, smooth=1e-6):
    preds = (torch.sigmoid(logits) >= threshold).float()
    if preds.shape[-2:] != targets.shape[-2:]:
        targets = F.interpolate(targets, size=preds.shape[-2:], mode='nearest')
    targets = targets.float()
    inter = (preds*targets).sum((1,2,3))
    denom = preds.sum((1,2,3)) + targets.sum((1,2,3))
    return ((2*inter+smooth)/(denom+smooth)).mean().item()


def set_encoder_mode(model, mode):
    # mode='frozen', 'last2', or 'all'
    for p in model.encoder.parameters():
        p.requires_grad = False
    if mode == 'last2':
        for block in model.encoder.blocks[-2:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in model.encoder.neck.parameters():
            p.requires_grad = True
    elif mode == 'all':
        for p in model.encoder.parameters():
            p.requires_grad = True


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    if not training:
        model.eval()
    total_loss = total_dice = 0.0
    n = 0
    for images, masks in loader:
        images, masks = images.to(DEVICE, non_blocking=True), masks.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = focal_tversky_loss(logits, masks)
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        total_loss += loss.item(); total_dice += dice_from_logits(logits.detach(), masks); n += 1
    return total_loss/max(n,1), total_dice/max(n,1)


def fit(model, train_loader, val_loader, epochs, lr, checkpoint, encoder_mode):
    set_encoder_mode(model, encoder_mode)
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.2, patience=4)
    best = float('inf'); bad = 0; patience = 8
    history = {'train_loss':[], 'val_loss':[], 'train_dice':[], 'val_dice':[], 'lr':[]}

    for epoch in range(1, epochs+1):
        tr_loss, tr_dice = run_epoch(model, train_loader, optimizer)
        va_loss, va_dice = run_epoch(model, val_loader)
        scheduler.step(va_loss)
        lr_now = optimizer.param_groups[0]['lr']
        for k,v in [('train_loss',tr_loss),('val_loss',va_loss),('train_dice',tr_dice),('val_dice',va_dice),('lr',lr_now)]: history[k].append(v)
        print(f'Epoch {epoch:02d}/{epochs} | train loss {tr_loss:.4f} dice {tr_dice:.4f} | val loss {va_loss:.4f} dice {va_dice:.4f} | lr {lr_now:.2e}')
        if va_loss < best - 1e-4:
            best = va_loss; bad = 0
            torch.save({'model_state_dict':model.state_dict(),'best_val_loss':best,'history':history}, checkpoint)
        else:
            bad += 1
            if bad >= patience:
                print('Early stopping.')
                break
    ckpt = torch.load(checkpoint, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    return history


## 7. Stage 1 — train only the new decoder

The medical-pretrained MedSAM encoder is frozen. This is the cheap transfer-learning stage.

In [9]:
STAGE1_CHECKPOINT = '/kaggle/working/medsam_transformer_unet_stage1.pt'
h1 = fit(model, train_loader, val_loader, epochs=15, lr=1e-3,
         checkpoint=STAGE1_CHECKPOINT, encoder_mode='frozen')


Epoch 01/15 | train loss 0.8188 dice 0.2022 | val loss 0.4009 dice 0.6295 | lr 1.00e-03
Epoch 02/15 | train loss 0.7736 dice 0.2460 | val loss 0.3232 dice 0.7281 | lr 1.00e-03
Epoch 03/15 | train loss 0.7604 dice 0.2604 | val loss 0.3563 dice 0.7108 | lr 1.00e-03
Epoch 04/15 | train loss 0.7585 dice 0.2624 | val loss 0.3884 dice 0.6884 | lr 1.00e-03
Epoch 05/15 | train loss 0.7536 dice 0.2670 | val loss 0.3125 dice 0.7489 | lr 1.00e-03
Epoch 06/15 | train loss 0.7401 dice 0.2810 | val loss 0.3542 dice 0.7186 | lr 1.00e-03
Epoch 07/15 | train loss 0.7436 dice 0.2768 | val loss 0.3369 dice 0.7280 | lr 1.00e-03
Epoch 09/15 | train loss 0.7390 dice 0.2821 | val loss 0.3468 dice 0.7417 | lr 1.00e-03
Epoch 10/15 | train loss 0.7395 dice 0.2815 | val loss 0.3899 dice 0.7068 | lr 2.00e-04
Epoch 11/15 | train loss 0.7385 dice 0.2823 | val loss 0.3743 dice 0.7053 | lr 2.00e-04
Epoch 12/15 | train loss 0.7291 dice 0.2912 | val loss 0.3151 dice 0.7545 | lr 2.00e-04
Epoch 13/15 | train loss 0.7268 

## 8. Stage 2 — fine-tune the last two ViT blocks + decoder

We do not fine-tune the whole 86M-parameter encoder immediately. Only the final transformer blocks and neck are unfrozen. This is intended to adapt medical-pretrained features to the specific FLAIR tumor task while limiting overfitting and GPU cost.

In [11]:
STAGE2_CHECKPOINT = '/kaggle/working/medsam_transformer_unet_best.pt'
h2 = fit(model, train_loader, val_loader, epochs=20, lr=1e-2,
         checkpoint=STAGE2_CHECKPOINT, encoder_mode='last2')


Epoch 01/20 | train loss 0.7943 dice 0.2259 | val loss 0.4366 dice 0.5853 | lr 1.00e-02
Epoch 02/20 | train loss 0.7721 dice 0.2486 | val loss 0.3887 dice 0.6625 | lr 1.00e-02
Epoch 03/20 | train loss 0.7780 dice 0.2411 | val loss 0.3920 dice 0.6635 | lr 1.00e-02
Epoch 04/20 | train loss 0.7667 dice 0.2526 | val loss 0.3827 dice 0.6483 | lr 1.00e-02
Epoch 05/20 | train loss 0.7580 dice 0.2621 | val loss 0.3976 dice 0.6180 | lr 1.00e-02
Epoch 07/20 | train loss 0.7465 dice 0.2743 | val loss 0.3671 dice 0.6860 | lr 1.00e-02
Epoch 08/20 | train loss 0.7760 dice 0.2404 | val loss 0.3782 dice 0.6906 | lr 1.00e-02
Epoch 09/20 | train loss 0.7436 dice 0.2760 | val loss 0.4175 dice 0.6448 | lr 1.00e-02
Epoch 10/20 | train loss 0.7670 dice 0.2516 | val loss 0.3422 dice 0.7175 | lr 1.00e-02
Epoch 11/20 | train loss 0.7481 dice 0.2729 | val loss 0.4020 dice 0.6681 | lr 1.00e-02
Epoch 12/20 | train loss 0.7385 dice 0.2830 | val loss 0.4417 dice 0.6359 | lr 1.00e-02
Epoch 13/20 | train loss 0.7557 

In [1]:
# Plot training history
history = {k: h1.get(k, []) + h2.get(k, []) for k in h1.keys()}
fig, ax = plt.subplots(1,2, figsize=(13,4))
ax[0].plot(history['train_loss'], label='train')
ax[0].plot(history['val_loss'], label='val')
ax[0].set_title('Loss'); ax[0].legend()
ax[1].plot(history['train_dice'], label='train')
ax[1].plot(history['val_dice'], label='val')
ax[1].set_title('Dice'); ax[1].legend()
plt.show()


NameError: name 'h1' is not defined

## 9. Test evaluation

Use the same fixed 0.5 threshold as the earlier models for the first comparison. We will not tune the threshold until after we know whether the architecture itself helps.

In [ ]:
@torch.no_grad()
def predict_dataframe(model, dataframe, threshold=0.5):
    model.eval(); rows=[]
    for _, row in dataframe.reset_index(drop=True).iterrows():
        img = io.imread(row.image_path)
        img = cv2.resize(img, (ENCODER_SIZE, ENCODER_SIZE), interpolation=cv2.INTER_LINEAR).astype(np.float32)
        if img.ndim == 2: img = np.repeat(img[...,None], 3, axis=-1)
        img = (img-img.min())/max(float(img.max()-img.min()),1e-6)
        tensor = torch.from_numpy(img.transpose(2,0,1)).float().unsqueeze(0).to(DEVICE)
        prob = torch.sigmoid(model(tensor))[0,0].cpu().numpy()
        pred = (prob >= threshold).astype(np.uint8)
        rows.append({'image_path':row.image_path,'mask_path':row.mask_path,'predicted_prob':prob,'predicted_mask':pred})
    return pd.DataFrame(rows)

df_pred = predict_dataframe(model, X_test, threshold=0.5)


def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true=y_true.astype(np.float64).flatten(); y_pred=y_pred.astype(np.float64).flatten()
    inter=np.sum(y_true*y_pred)
    return (2*inter+smooth)/(np.sum(y_true)+np.sum(y_pred)+smooth)

def iou_coef(y_true,y_pred,smooth=1e-6):
    y_true=y_true.astype(np.float64).flatten(); y_pred=y_pred.astype(np.float64).flatten()
    inter=np.sum(y_true*y_pred); union=np.sum(y_true)+np.sum(y_pred)-inter
    return (inter+smooth)/(union+smooth)

def precision_recall(y_true,y_pred):
    y_true=y_true.flatten().astype(np.uint8); y_pred=y_pred.flatten().astype(np.uint8)
    tp=np.sum((y_true==1)&(y_pred==1)); fp=np.sum((y_true==0)&(y_pred==1)); fn=np.sum((y_true==1)&(y_pred==0))
    return tp/max(tp+fp,1), tp/max(tp+fn,1)

results=[]
for _,r in df_pred.iterrows():
    gt=cv2.resize(io.imread(r.mask_path),(OUTPUT_SIZE,OUTPUT_SIZE),interpolation=cv2.INTER_NEAREST)
    gt=(gt>0).astype(np.uint8); pred=r.predicted_mask.astype(np.uint8)
    p,rec=precision_recall(gt,pred)
    results.append({'image_path':r.image_path,'dice':dice_coef(gt,pred),'iou':iou_coef(gt,pred),
                    'pixel_accuracy':np.mean(gt==pred),'precision':p,'recall':rec})
metrics_df=pd.DataFrame(results)
print(metrics_df[['dice','iou','pixel_accuracy','precision','recall']].agg(['mean','std']))
print('
Mean metrics:')
print(metrics_df[['dice','iou','pixel_accuracy','precision','recall']].mean())


## 10. Error analysis

In [ ]:
analysis_rows=[]
for _,r in df_pred.iterrows():
    gt=cv2.resize(io.imread(r.mask_path),(OUTPUT_SIZE,OUTPUT_SIZE),interpolation=cv2.INTER_NEAREST)
    gt=(gt>0).astype(np.uint8); pred=r.predicted_mask.astype(np.uint8)
    gf,pf=gt.flatten(),pred.flatten()
    tp=int(np.sum((gf==1)&(pf==1))); fp=int(np.sum((gf==0)&(pf==1))); fn=int(np.sum((gf==1)&(pf==0)))
    analysis_rows.append({'image_path':r.image_path,'tp':tp,'fp':fp,'fn':fn,'gt_area':int(gf.sum()),'pred_area':int(pf.sum())})
analysis_df=pd.DataFrame(analysis_rows).merge(metrics_df,on='image_path')
print('False negative pixels:', int(analysis_df.fn.sum()))
print('False positive pixels:', int(analysis_df.fp.sum()))
print('FN/FP ratio:', round(int(analysis_df.fn.sum())/max(int(analysis_df.fp.sum()),1),2))


In [ ]:
# Qualitative examples
n = min(6, len(df_pred))
fig, axs = plt.subplots(n, 3, figsize=(12, 4*n))
if n == 1: axs = np.expand_dims(axs,0)
for i in range(n):
    r=df_pred.iloc[i]
    img=io.imread(r.image_path)
    gt=cv2.resize(io.imread(r.mask_path),(OUTPUT_SIZE,OUTPUT_SIZE),interpolation=cv2.INTER_NEAREST)>0
    pred=r.predicted_mask.astype(bool)
    axs[i,0].imshow(img); axs[i,0].set_title('MRI')
    axs[i,1].imshow(img); axs[i,1].imshow(gt,alpha=.45); axs[i,1].set_title('Ground truth')
    axs[i,2].imshow(img); axs[i,2].imshow(pred,alpha=.45); axs[i,2].set_title('MedSAM-Transformer prediction')
    for j in range(3): axs[i,j].axis('off')
plt.tight_layout(); plt.show()


## Next experiments

If this model beats the ~0.821 Swin-Tiny result on the same positive-only split, keep it as the new architecture candidate and then run controlled experiments:

1. **Threshold tuning** on validation only.
2. **Focal Tversky alpha sweep** (0.5 / 0.6 / 0.7 / 0.8).
3. Compare **Dice + BCE**.
4. Add elastic augmentation.
5. Then test a **patient-level split**.
6. Finally, if the model is strong, add TTA / small connected-component post-processing.

Do not use the ground-truth bounding box as a test prompt if the goal is an automatic segmentation system. MedSAM's standard interface is box-prompted; using the true test box would answer a different, interactive-segmentation question. citeturn0search0turn1search5